# Tethys-Chloris (T&C): model setup and sensitivity analysis of soil moisture dynamics

## Material
* Case Study Materials
  * Script to create the model inputs

* Input Data
  * Files containing CO2 air concentration

* Visualization Tools
  * Functions for results visualizations

## Objectives

This session is designed to help students learn how to run a basic case study using the plot-scale version of the T&C model (*Fatichi et al., 2012*) with a specific focus of analyzing soil moisture dynamics under variable inputs and conditions. The document outlines the key steps, verifying critical model settings, understanding model inputs and outputs, and conducting model sensitivity analysis of soil moisture dynamics. 

The tutorial here will focus on one case study from Switzerland ([Chamau, ZG](https://www.google.com/maps/place/47%C2%B012'36.8%22N+8%C2%B024'38.3%22E/@47.2102278,8.4099778,273m/data=!3m2!1e3!4b1!4m4!3m3!8m2!3d47.2102269!4d8.4106451?entry=ttu)).

# Part A: input preparation, model setup and validation

## 1. Julia installation

Please make sure you have [installed Julia](https://julialang.org/downloads/) on your computer.

Once this is done, install Julia 1.12, `cd` to the main directory of this repository (the one containing the `Project.toml` file) and start Julia in your terminal using

```bash
juliaup add 1.12
cd <main-directory>
julia +1.12
```

Then. install the dependencies associated with the exercises by running the following commands in the REPL

```julia
julia> ]
(@v1.12) pkg> activate .
(julia) pkg> instantiate
```

After this is done, select the Julia 1.12 channel kernel to run this notebook. **All subsequent steps can be run from the notebook**

## 2. Overview of the model

The implementation of the Tethys-Chloris model runs as follows
1. **Initialize the model state** at the first iteration, based on the model parameters, initial conditions, as well as forcing input
2. **Run the simulation** for a specific number of iterations
3. **Analyse the results** using the diagnostics output by the simulation as well as the figures provide by the package


## 3. Preparing the outputs

The Tethys-Chloris model requires two specific input files to run the model: a configuration file and a forcing file. These files are essential for defining the model's parameters and the environmental conditions under which it operates.

For this exercise, we provide you two helper functions , `prepare_parameters` and `prepare_inputs`, to easily create the configuration and forcing files.

In [ ]:
include(joinpath(@__DIR__, "..", "utils", "data_utils.jl"))
include(joinpath(@__DIR__, "prepare_data.jl"))

### 3.1 Configuration (model parameters)
The configuration file is a YAML file that contains various parameters necessary for the model's operation. The dictionary will be used to initialize the model parameters. 

Below, we use the `prepare_parameters` helper function to create the model parameters

In [ ]:
params = prepare_parameters()

Subsequently, we save the parameters to the YAML file which will be read by the model initialization

In [ ]:
save_parameters(params, joinpath(@__DIR__, "data", "CH-Cha.yaml"))

### 3.2 Meteorological forcing data
The forcing file is a NetCDF file that provides the environmental data needed for the model, which tonains the following variables, at an hourly scale
* Date and time
* Temperature (Ta)
* Precipitation (Pr)
* Wind speed (Ws)
* Vapor pressure (ea)
* Solar radiation components (SAD1, SAD2, SAB1, SAB2)
* Dew point temperature (Tdew)
* Saturation vapor pressure (esat)
* Photosynthetically active radiation (PARB, PARD)
* CO2 concentration (Ca)

In [ ]:
forcing_input = joinpath(@__DIR__, "data", "Data_CH-Cha_run.mat")
ca_path = joinpath(@__DIR__, "data", "Ca_Data.mat")
netcdf_path = joinpath(@__DIR__, "data", "CH-Cha.nc")

forcing = prepare_netcdf(
  forcing_input,
  ca_path,
  netcdf_path
)

## 4. Understand the meteorological input file

Meteorological input data, such as air temperature, precipitation, and solar radiation, are essential inputs to the ecohydrological model T&C. To prepare the input data required for T&C, we will use data from the [FLUXNET monitoring network](https://fluxnet.org/). Specifically, the Swiss FluxNet, a regional subset of FLUXNET, currently encompasses six long-term ecosystem monitoring sites in Switzerland. The original data has been pre-processed to fit the input format of T&C in Chamau.

* The file **`CH-Cha.nc`** contains the necessary meteorological forcing to run T&C. To understand the meaning of each variable in these files, refer to the **`T&C_Variables_LIST_PlotScale.pdf`** or the slides.

Once you have created the file, consider the following questions:

- *What meteorological data does the file contain?*
- *What is the time period covered by the meteorological data?*
- *What are the mean annual precipitation and mean temperature in Chamau?*
- *To which biome does this site belong according to the Whittaker classification (Whittaker, 1970)?*

### Accessing data stored in the NetCDF file in Julia

To access the file, you will need to load the [`NCDatasets`](https://juliageo.org/NCDatasets.jl/stable/) package in the environment and open the file as follows

```julia
using NCDatasets
ds = NCDataset(netcdf_path, "r");
```

The input file will then be accessible in read-only (`"r"`) mode via the variable `ds`.

The data is stored in the [NetCDF](https://www.unidata.ucar.edu/software/netcdf)(Network Common Data Form) format, which provides convenient information about the dimensions of all variables. To access a variable, such as the wind speed, 

```julia
ds["Ws"]
```

## 5. Understand the case study

Open the file **`prepare_data.jl`** and take a moment to familiarize yourself with the parameters of the case study, defined in the function `prepare_parameters`:

- *What is the vegetation type and land cover in Chamau? Is there any land management?*
- *What are the soil depth and soil texture? How many soil layers are set?*

*Hint*: To get a better idea about the vegetation in Chamau, check the variables **ZR95** and **aSE** for the high and low vegetation, **Ccrown**, and **vegetationmanagement** for the low vegetation. The abbreviations and meanings of these variables can be found in **`T&C_Variables_LIST_PlotScale.pdf`**.

Remember to perform the same checks before running the simulation for a new case study. For the case study in Chamau all parameters are already provided.

## 6. Ready to run the simulation

### 6.1 Initializing the model 

To run the model, initializing the model based on the parameters, forcing inputs and initial conditions. 

⚠️ *Runnning this cell for the first time can take some time due to the compilation of the TethysChloris.jl functions* ⚠️ 

In [ ]:
using TethysChloris

yaml_path = joinpath(@__DIR__, "data", "CH-Cha.yaml");
netcdf_path = joinpath(@__DIR__, "data", "CH-Cha.nc");

# Initialize model
FT = Float64
model = initialize_model(FT, netcdf_path, yaml_path);

### 6.2 Specifying the options for the simulation

Then, specify the options for the simulation. These options allow the user to fine-tune the simulation by enabling or disabling some components of the model, as well as by specifying the root-finding method or ODE solver used for specific functions.

 By default, the `SoilTemperature`, `FreezingSoil` and `VegetationSnowInteractions` options are disabled, but for Chamau, we would like to turn them on. 

 Additionally, for stability, we would like to use the [`Order0`](https://juliamath.github.io/Roots.jl/stable/reference/#Roots.Order0) method to find the zero of the canopy resistance function, and to use [Brent's method](https://en.wikipedia.org/wiki/Brent%27s_method) to solve the surface temperature.

In [ ]:
import Roots

options = ModelOptions(
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT, Roots.Order0()),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
);

Now that everything is ready, you can run the simulation

In [ ]:
diagnostics = run_simulation(model; NN = 87648, options);

The results of the simulation are stored in the `model` structure, which contains the staet of all variables at each time step.

The simulation outputs four diagnostic metrics
* `ALB`: the surface albedo at each timestep [-]
* `CK1`: the water budget residual at each timestep [mm/h]
* `Ck`: the water mass balance residual over the entire simulation [mm/h]
* `DQ`: the residual of the energy budget [W/m^2]

These metrics are helpful to assess whether the simulation provides reasonable results, or whether something is amiss with the results. 

Before proceeding to checking the results of the simulation, check that the water budget residual at each time step and over the entire simulation are reasonable

```julia
using Plots
plot(diagnostics[2], label = "Water budget residual", ylims = (-1e-9, 1e-9))
```

### 6.3 Result checking

To quickly check some results, you can easily plot variables. For example, try plotting the **O (volumetric soil water content)** using the `plot` function. The `O` variable is a hydrologic state variable, which can be accessed from the model as `model.state.hydrologic.O`.

```julia
using Plots
plot(model.state.hydrologic.O)
```

To get an overview of the results, use the provided plotting functions provided in the [TethysChloris package](https://epfl-enac.github.io/change-tethyschloris-doc/stable/04-plots/). 

# Part B: Sensitivity analysis

In this second part, we will perform a sensitivity analysis of model outputs (particularly soil moisture dynamics – Lecture 2) to variations in soil texture, root depth, and precipitation.

## 1. Sensitivity analysis

Modify the parameters listed in Table 1 and explore how these parameters impact key ecohydrological variables. See Table 2 for an overview of the most important variables. Note that, for some variables, such as runoff, cumulative values might be more informative than means. For illustrative purposes you may consider a sandy soil when varying vegetation properties and precipitation, even though it does not reflect the true condition at the site.

In order to modify the soil properties, you can pass the values of your choosing to `prepare_parameters` via the `Psan`and `Pcla` arguments. By default, these values are 0.254 and 0.244, respectively. Similary, you can use the `ZR95_L` and `Pre_frac` keyword arguments of `prepare_netcdf` to modify the vegetation properties and precipitation. Remember to check the `prepare_data.jl` file for the types expected by these functions.

After modifying the parameters, remember to initialize the model before running the simulation.


### Effect of soil and vegetation properties, and precipitation on soil moisture dynamics

#### Soil properties

| Variables                           | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| ----------------------------------- | ----- | -------------- | --------- | --------- | ------- | 
| (Sand) Psan=80%, Pcla=5%            | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| (Silty loam) Psan=25.4%, Pcla=24.4%  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| (Clay) Psan=25%, Pcla=50%           | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 

#### Vegetation properties

| Variables                               | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| --------------------------------------- | ----- | -------------- | --------- | --------- | ------- | 
| Root depth decrease to 100 [mm] (sand)  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| Root depth increase to 500 [mm] (sand)  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 

#### Precipitation

| Variables     | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| ------------- | ----- | -------------- | --------- | --------- | ------- | 
| Double (sand) | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| Half (sand)   | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; |

### Relevant state variables in Tethys-Chloris

| Name | Unit | Description |
| ------------- | ----- | -------------- |
| O | [-] | Soil moisture – Liquid volumetric soil water content |
| Osat | [-] | Water content at saturation | 
| ET | [mm/h] | Total evapotranspiration | 
| Lk | [mm/h] | Bottom leakage soil to bedrock | 
| Rd | [mm/h] | Saturation excess runoff | 
| Rh | [mm/h] | Infiltration excess runoff | 



## 2.  Interpreting and plotting the results
In order to get a better understanding of the soil moisture dynamics, let’s take a closer look at how the variables above evolve over time.
* Plot the soil moisture in time at different depths, as well as ET, Lk, and Rh. How do these variables relate to each other?
* Plot the time-averaged soil moisture at different depths. Can you explain the differences between the scenarios from the sensitivity analysis? For instance, how does soil texture affect moisture, and why? Is there anything unexpected? You can use the code provided in `prepare_plots.m`.
* Can you compute for the different cases how much water you have in the root zone? How does this change with root depth/precipitation/soil type?

### Find out more

[Laboratory of Catchment Hydrology and Geomorphology](https://www.epfl.ch/labs/change/)
[T&C source code (Julia)](github.com/CHANGE-EPFL/TethysChloris.jl)
[T&C source code (MATLAB)](https://github.com/simonefatichi/TeC_Source_Code)


### References 

Fatichi, S., Ivanov, V. Y., & Caporali, E. (2012). A mechanistic ecohydrological model to investigate complex interactions in cold and warm water‐controlled environments: 1. Theoretical framework and plot‐scale analysis. *Journal of Advances in Modeling Earth Systems, 4*(2).
Fatichi, S., Zeeman, M. J., Fuhrer, J., & Burlando, P. (2014). Ecohydrological effects of management on subalpine grasslands: From local to catchment scale. *Water Resources Research, 50*(1), 148-164